# Aerial OBB Object Detection & Benchmark Suite
### Google Colab GPU Runner with Google Drive Persistence & Crash Resumption

Run this notebook in Google Colab with a **GPU runtime**:
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU (or A100/V100/L4)`

This pipeline benchmarks 10 state-of-the-art post-2022 Aerial OBB architectures across datasets:
1. **Models**: `std`, `rvsa`, `ars-detr`, `oriented-former`, `rio-detr`, `rhino`, `ao2-detr`, `swin-obb`, `lsknet`, `yolo11-obb`
2. **Datasets**: `VisDrone`, `CODrone`, `DOTA` (persisted on Google Drive)
3. **Google Drive Integration**: Automatically mounts `/content/drive/MyDrive/object-detection` for persistent storage of datasets (`data/`), model weights (`weights/`), and outputs (`results/`).
4. **Full 4-Stage Pipeline**: Pre-eval baseline -> GPU Training (5 Epochs) -> Post-eval on fine-tuned weights -> Comparative Delta Synthesis (ΔmAP50, ΔF1, ΔAngle MAE) & Visual Diagnostic Plots.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Initialize Google Drive directory structure for object-detection
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/object-detection")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_RESULTS = DRIVE_ROOT / "results"

for folder in [DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir : {DRIVE_DATA}")
print(f"    - Weights Dir  : {DRIVE_WEIGHTS}")
print(f"    - Results Dir  : {DRIVE_RESULTS}")

In [ ]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Suyash8/minor-project.git"
if not os.path.exists("/content/minor-project"):
    !git clone $REPO_URL /content/minor-project
%cd /content/minor-project
!git pull

In [ ]:
# 5. Install required packages (Ultralytics, Shapely, Scipy, etc.)
!python install.py

In [ ]:
# 6. Dataset Linking, Bundled Annotation Unpacking & Verification
# Google Drive datasets (/content/drive/MyDrive/object-detection/data/)
# are symlinked to local data/ for zero-copy high-throughput I/O.
import os
import zipfile
from pathlib import Path
from src.data.downloader import verify_dataset_status

# 1. Link Drive datasets to local data/ directory
!mkdir -p data
drive_data = Path("/content/drive/MyDrive/object-detection/data")
if drive_data.exists():
    for d in drive_data.iterdir():
        target = Path(f"data/{d.name}")
        if not target.exists():
            os.symlink(str(d), str(target))
            print(f"[✓] Linked {d.name} from Google Drive")

# 2. Auto-unpack bundled annotations if Drive/local datasets lack labels
for d_name in ["visdrone", "codrone", "dota"]:
    zpath = Path(f"assets/{d_name}_annotations.zip")
    target_d = Path(f"data/{d_name}")
    if zpath.exists() and target_d.exists():
        has_txt = any(target_d.rglob("*.txt"))
        has_xml = any(target_d.rglob("*.xml"))
        if not has_txt and not has_xml:
            print(f"[*] Unpacking bundled annotations for {d_name} from {zpath.name}...")
            with zipfile.ZipFile(zpath, "r") as z:
                z.extractall(target_d)
            print(f"[✓] Annotations unpacked successfully for {d_name}.")

# 3. Verify dataset readiness
print("\n--- DATASET STATUS CHECK ---")
for d in ["visdrone", "codrone", "dota"]:
    status = verify_dataset_status(d, Path("data"))
    if status["ready"]:
        print(f"  [{d.upper():<8}] [READY] Found {status['num_images']} images across splits: {status['splits_found']}")
    else:
        print(f"  [{d.upper():<8}] [MISSING / INCOMPLETE] (Will attempt download or check drive)")


In [ ]:
# 7. Fast Smoke Test (Optional: verifies all 10 models in fast test mode in ~30 seconds)
!python scripts/run_pipeline.py \
    --datasets visdrone \
    --models std rvsa ars-detr oriented-former rio-detr rhino ao2-detr swin-obb lsknet yolo11-obb \
    --mode full \
    --test \
    --save-plots \
    --run-name smoke_test_all_10


In [ ]:
# 8. Full End-to-End GPU Pipeline: ALL 10 Models across ALL Datasets (5 Epochs)
# Stage 1: Pre-Training Baseline Evaluation (measures zero-shot / base weights)
# Stage 2: GPU Training & Fine-Tuning Suite (5 epochs per model/dataset combination on CUDA)
# Stage 3: Post-Training Evaluation (evaluates fine-tuned checkpoints on validation set)
# Stage 4: Comparative Delta Synthesis (computes exact empirical gains: ΔmAP50, ΔF1, ΔAngle MAE)
#
# Models: std, rvsa, ars-detr, oriented-former, rio-detr, rhino, ao2-detr, swin-obb, lsknet, yolo11-obb
# Datasets: visdrone, codrone, dota
!python scripts/run_pipeline.py \
    --mode full \
    --models std rvsa ars-detr oriented-former rio-detr rhino ao2-detr swin-obb lsknet yolo11-obb \
    --datasets visdrone codrone dota \
    --epochs 5 \
    --train-batch-size 8 \
    --batch-size 8 \
    --train-workers 2 \
    --device cuda \
    --save-plots \
    --resume True \
    --download \
    --run-name full_train_all_5epochs


In [ ]:
# 9. Display Generated Benchmark Report & Comparative Progression Plots Inline
import os
import glob
from pathlib import Path
from IPython.display import Image, display, Markdown

# Search in Google Drive results first, then local fallback
search_dirs = ["/content/drive/MyDrive/object-detection/results/*", "results/*"]
run_dirs = []
for pattern in search_dirs:
    for p in glob.glob(pattern):
        if os.path.isdir(p) and not p.endswith(".tmp"):
            run_dirs.append(p)

run_dirs = sorted(set(run_dirs))
if run_dirs:
    latest_run = run_dirs[-1]
    print(f"Latest Run Directory: {latest_run}")
    report_file = os.path.join(latest_run, "benchmark_report.md")
    if os.path.exists(report_file):
        with open(report_file) as f:
            display(Markdown(f.read()))

    # 1. Display Before-vs-After Gain Progression Chart
    pre_post_chart = os.path.join(latest_run, "plots", "pre_vs_post_comparison.png")
    if os.path.exists(pre_post_chart):
        print("\n=======================================================")
        print("--- Pre-Training vs Post-Training Performance Gain ---")
        print("=======================================================")
        display(Image(filename=pre_post_chart))

    # 2. Display Overall Model Benchmark Comparison Chart
    comparison_chart = os.path.join(latest_run, "plots", "model_benchmark_comparison.png")
    if os.path.exists(comparison_chart):
        print("\n--- Overall Model Benchmark Comparison ---")
        display(Image(filename=comparison_chart))

    # 3. Display Confusion Matrices and Correlation Plots
    for img_path in sorted(glob.glob(f"{latest_run}/plots/*.png")):
        bname = os.path.basename(img_path)
        if bname not in ("model_benchmark_comparison.png", "pre_vs_post_comparison.png"):
            print(f"\nDisplaying: {bname}")
            display(Image(filename=img_path))
else:
    print("No results directory found yet. Run step 8 above first.")
